# Movie Expert Agent (OpenAI Function Calling)

This notebook builds a Movie Expert Agent with OpenAI function calling and a real movie API.


In [1]:
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI


In [2]:
load_dotenv()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def _get_json(path: str):
    response = requests.get(f"{BASE_URL}{path}", timeout=15)
    response.raise_for_status()
    return response.json()

def get_popular_movies():
    """Fetch popular movies from /movies."""
    return _get_json("/movies")

def get_movie_details(id: int):
    """Fetch movie details from /movies/:id."""
    return _get_json(f"/movies/{id}")

def get_movie_credits(id: int):
    """Fetch cast and crew from /movies/:id/credits."""
    return _get_json(f"/movies/{id}/credits")


In [3]:
tools = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Use this when the user asks for currently popular or trending movies.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Use this when the user asks what movie corresponds to a specific movie ID, or asks for movie details by ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {"type": "integer", "description": "Movie ID to look up"}
            },
            "required": ["id"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "get_movie_credits",
        "description": "Use this when the user asks who appears in a movie, or asks for cast/crew by movie ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {"type": "integer", "description": "Movie ID to look up"}
            },
            "required": ["id"],
            "additionalProperties": False
        }
    }
]

SYSTEM_PROMPT = """
You are a Movie Expert Agent.
Always choose exactly one function call from the provided tools.

Tool rules:
1) Use get_popular_movies() for popular/trending movie requests.
2) Use get_movie_details(id) for questions asking what movie corresponds to an ID, or for details by ID.
3) Use get_movie_credits(id) for cast/crew questions by ID.

Argument rules:
- If a movie ID appears in the user message, pass it as an integer in `id`.
- For popular movie requests, call get_popular_movies with no arguments.
- Respond with a function call.
""".strip()

client = OpenAI()


In [4]:
def run_selected_tool(tool_name: str, args: dict):
    if tool_name == "get_popular_movies":
        return get_popular_movies()
    if tool_name == "get_movie_details":
        return get_movie_details(int(args["id"]))
    if tool_name == "get_movie_credits":
        return get_movie_credits(int(args["id"]))
    raise ValueError(f"Unknown tool: {tool_name}")

def select_and_execute(user_message: str):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {
                "role": "system",
                "content": [{"type": "input_text", "text": SYSTEM_PROMPT}],
            },
            {
                "role": "user",
                "content": [{"type": "input_text", "text": user_message}],
            },
        ],
        tools=tools,
    )

    function_call = next((item for item in response.output if item.type == "function_call"), None)
    if function_call is None:
        raise RuntimeError("Model did not produce a function call.")

    tool_name = function_call.name
    args = json.loads(function_call.arguments or "{}")

    print(f"[Selected function] {tool_name}")
    print(f"[Arguments] {args}")

    tool_result = run_selected_tool(tool_name, args)
    return tool_name, args, tool_result


In [5]:
test_inputs = [
    "Tell me what movies are popular right now",
    "Tell me what movie corresponds to movie ID 550",
    "Tell me who appears in the movie with movie ID 550",
]

for idx, text in enumerate(test_inputs, start=1):
    print(f"\n===== Test {idx} =====")
    print(f"Question: {text}")
    name, args, result = select_and_execute(text)

    if isinstance(result, list):
        print(f"[API result] list length: {len(result)}")
        if result:
            print(f"[Sample item] {result[0]}")
    elif isinstance(result, dict):
        print(f"[API result keys] {list(result.keys())[:10]}")
        if "title" in result:
            print(f"[Movie title] {result['title']}")
        if "cast" in result:
            cast_names = [c.get('name') for c in result.get('cast', [])[:5]]
            print(f"[Top 5 cast] {cast_names}")
    else:
        print(f"[API result] {result}")



===== Test 1 =====
Question: Tell me what movies are popular right now
[Selected function] get_popular_movies
[Arguments] {}
[API result] list length: 20
[Sample item] {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/1x9e0qWonw634NhIsRdvnneeqvN.jpg', 'genre_ids': [10749, 18], 'id': 1523145, 'original_language': 'ru', 'original_title': 'Твоё сердце будет разбито', 'overview': 'High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple develops real feelings, but her family and classmates have reasons to separate the lovers.', 'popularity': 1192.7521, 'poster_path': 'https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg', 'release_date': '2026-03-26', 'title': 'Your Heart Will Be Broken', 'video': False, 'vote_average': 6.602, 'vote_count': 49}

===== Test 2 =====
Question: Tell me what movi